In [1]:
import sqlite3
import pandas as pd 

In [11]:
conn = sqlite3.connect('../data/processed/telemetry.db')

tables = pd.read_sql("PRAGMA table_list;", conn)

for idx, row in tables.iterrows():
    print(f"{idx}: {row['name']} ({row['ncol']} columns)")

0: function_durations_percentiles.anon.d09 (14 columns)
1: function_durations_percentiles.anon.d08 (14 columns)
2: app_memory_percentiles.anon.d09 (12 columns)
3: app_memory_percentiles.anon.d08 (12 columns)
4: invocations_per_function_md.anon.d03 (1444 columns)
5: app_memory_percentiles.anon.d01 (12 columns)
6: invocations_per_function_md.anon.d02 (1444 columns)
7: app_memory_percentiles.anon.d07 (12 columns)
8: function_durations_percentiles.anon.d02 (14 columns)
9: invocations_per_function_md.anon.d05 (1444 columns)
10: function_durations_percentiles.anon.d11 (14 columns)
11: invocations_per_function_md.anon.d14 (1444 columns)
12: function_durations_percentiles.anon.d10 (14 columns)
13: invocations_per_function_md.anon.d13 (1444 columns)
14: function_durations_percentiles.anon.d01 (14 columns)
15: invocations_per_function_md.anon.d04 (1444 columns)
16: app_memory_percentiles.anon.d06 (12 columns)
17: function_durations_percentiles.anon.d13 (14 columns)
18: function_durations_percent

In [10]:
table_name = '"function_durations_percentiles.anon.d04"' 
schema_info = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
print(f"\nSchema for {table_name}:")
print(schema_info[['name', 'type']])


Schema for "function_durations_percentiles.anon.d04":
                      name     type
0                hashowner     TEXT
1                  hashapp     TEXT
2             hashfunction     TEXT
3                  average  INTEGER
4                    count  INTEGER
5                  minimum     REAL
6                  maximum     REAL
7     percentile_average_0  INTEGER
8     percentile_average_1  INTEGER
9    percentile_average_25  INTEGER
10   percentile_average_50  INTEGER
11   percentile_average_75  INTEGER
12   percentile_average_99  INTEGER
13  percentile_average_100  INTEGER


In [12]:
table_name = '"app_memory_percentiles.anon.d09"' 
schema_info = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
print(f"\nSchema for {table_name}:")
print(schema_info[['name', 'type']])


Schema for "app_memory_percentiles.anon.d09":
                         name     type
0                   hashowner     TEXT
1                     hashapp     TEXT
2                 samplecount  INTEGER
3          averageallocatedmb  INTEGER
4     averageallocatedmb_pct1  INTEGER
5     averageallocatedmb_pct5  INTEGER
6    averageallocatedmb_pct25  INTEGER
7    averageallocatedmb_pct50  INTEGER
8    averageallocatedmb_pct75  INTEGER
9    averageallocatedmb_pct95  INTEGER
10   averageallocatedmb_pct99  INTEGER
11  averageallocatedmb_pct100  INTEGER


In [13]:
table_name = '"invocations_per_function_md.anon.d04"' 
schema_info = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
print(f"\nSchema for {table_name}:")
print(schema_info[['name', 'type']])


Schema for "invocations_per_function_md.anon.d04":
              name     type
0        hashowner     TEXT
1          hashapp     TEXT
2     hashfunction     TEXT
3          trigger     TEXT
4                1  INTEGER
...            ...      ...
1439          1436  INTEGER
1440          1437  INTEGER
1441          1438  INTEGER
1442          1439  INTEGER
1443          1440  INTEGER

[1444 rows x 2 columns]


In [16]:
import matplotlib.pyplot as plt

# 1. Grab a single app's record for Day 1
query = 'SELECT * FROM "invocations_per_function_md.anon.d01" LIMIT 1'
df_wide = pd.read_sql(query, conn)

# 2. Melt the 1,440 columns into a single time-series
# We keep the identifiers and turn the '1', '2', '3' columns into a 'Minute' row
df_long = df_wide.melt(
    id_vars=['hashapp', 'hashfunction', 'trigger'], 
    var_name='minute', 
    value_name='invocations'
)

# 3. Convert 'minute' to numeric so it sorts correctly on the X-axis
df_long['minute'] = pd.to_numeric(df_long['minute'])
df_long = df_long.sort_values('minute')

# 4. Plot the "Demand Curve"
plt.figure(figsize=(12, 5))
plt.plot(df_long['minute'], df_long['invocations'])
plt.title(f"24-Hour Demand for App: {df_long['hashapp'].iloc[0][:8]}...")
plt.xlabel("Minute of Day (0-1440)")
plt.ylabel("Number of Invocations")
plt.grid(True, alpha=0.3)
plt.show()

ValueError: Unable to parse string "hashowner" at position 0